# 오늘의집 원룸 인테리어 이미지 크롤링

[오늘의집 피드](https://ohou.se/cards/feed?order=best&residence=0&color=grey) (역대 인기순 · 원룸&오피스텔 · 그레이)에서 로그인 없이 이미지를 수집합니다.

- **백그라운드 실행** (Chrome 최소화 — headless는 사이트에서 차단됨)
- **위에서부터 한 줄(4개)씩** 다운로드
- 화면에 더 받을 이미지가 없을 때만 스크롤

In [1]:
# 크롤링 의존성: selenium(브라우저), webdriver-manager(ChromeDriver), requests(이미지 다운로드)
%pip install selenium webdriver-manager requests

  Using cached webdriver_manager-4.1.2-py3-none-any.whl.metadata (16 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached cffi-2.0.0-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.meta

In [12]:
import re  # 썸네일 URL → 고해상도 변환
import time  # 스크롤·대기
import hashlib  # URL → 파일명 해시
from pathlib import Path  # 저장 경로
from urllib.parse import urlparse  # URL에서 확장자 추출

import requests  # HTTP 이미지 다운로드
from selenium import webdriver  # Chrome 브라우저 제어
from selenium.webdriver.chrome.service import Service  # ChromeDriver 서비스
from selenium.webdriver.chrome.options import Options  # headless/최소화 등 옵션
from selenium.webdriver.common.by import By  # CSS 셀렉터
from selenium.webdriver.support.ui import WebDriverWait  # 요소 로딩 대기
from selenium.webdriver.support import expected_conditions as EC  # 대기 조건
from webdriver_manager.chrome import ChromeDriverManager  # ChromeDriver 자동 설치

In [20]:
# === 설정 ===
SEARCH_URL = "https://ohou.se/cards/feed?order=best&residence=0&color=black"  # 오늘의집 피드 URL
SAVE_DIR = Path("images/원룸_인테리어_오늘의집")  # 이미지 저장 폴더
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CARD_IMG_SELECTOR = "article.card-item img.image"  # 콘텐츠 사진 (프로필 제외)
ROW_SIZE = 4           # 한 줄(행)당 카드 수
SCROLL_PAUSE = 2.0     # 스크롤 후 대기(초)
PAGE_LOAD_WAIT = 30    # 카드 로딩 대기(초)
MAX_IMAGES = 100       # 최대 다운로드 개수

In [14]:
def create_driver():
    """Chrome 드라이버 생성 (최소화 모드, 봇 탐지 우회)."""
    options = Options()
    # headless는 오늘의집에서 Access Denied → 최소화로 백그라운드 실행
    options.add_argument("--start-minimized")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),  # ChromeDriver 자동 설치
        options=options,
    )
    driver.execute_script(  # navigator.webdriver 숨김
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def dismiss_popups(driver):
    """로그인/앱 유도 팝업 등 닫기 시도."""
    for selector in (
        "button[aria-label='Close']",
        "button[aria-label='닫기']",
        "[class*='close']",
    ):
        try:
            for btn in driver.find_elements(By.CSS_SELECTOR, selector):
                if btn.is_displayed():
                    btn.click()
                    time.sleep(0.5)
        except Exception:
            pass


def wait_for_cards(driver, timeout=PAGE_LOAD_WAIT):
    """검색 결과 카드가 로드될 때까지 대기."""
    print(f"페이지 로딩 대기 중... (최대 {timeout}초)")
    dismiss_popups(driver)

    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, CARD_IMG_SELECTOR)
            )
        )
    except Exception:
        title = driver.title
        if "Access Denied" in title:
            raise RuntimeError(
                "오늘의집 접근 차단(Access Denied). "
                "headless가 아닌 최소화 모드인지, 인터넷 연결을 확인하세요."
            ) from None
        raise TimeoutError(
            f"{timeout}초 안에 검색 결과 카드를 찾지 못했습니다.\n"
            f"  페이지 제목: {title}\n"
            f"  현재 URL: {driver.current_url}\n"
            "  → Chrome/인터넷이 느리거나, 셀 실행 순서가 잘못됐을 수 있습니다.\n"
            "  → 커널 재시작 후 import → 설정 → 함수 → 실행 순서로 다시 실행하세요."
        ) from None

    time.sleep(1)
    n = len(driver.find_elements(By.CSS_SELECTOR, CARD_IMG_SELECTOR))
    print(f"카드 {n}개 확인")

In [15]:
def to_original_size(url: str) -> str:
    """썸네일 URL을 고해상도로 변환 (w=480 → w=1200)."""
    url = re.sub(r"w=\d+", "w=1200", url)
    url = re.sub(r"h=\d+", "h=1200", url)
    return url


def get_pending_urls(driver, downloaded):
    """위→아래 순서로, 아직 다운로드하지 않은 img.image src URL."""
    pending = []
    for img in driver.find_elements(By.CSS_SELECTOR, CARD_IMG_SELECTOR):
        src = img.get_attribute("src")
        if not src or not src.startswith("http"):
            continue
        url = to_original_size(src)
        if url not in downloaded and url not in pending:
            pending.append(url)
    return pending


def scroll_once(driver, pause=SCROLL_PAUSE):
    """한 칸 아래로 스크롤. 새 카드가 로드되면 True."""
    before = len(driver.find_elements(By.CSS_SELECTOR, CARD_IMG_SELECTOR))
    driver.execute_script("window.scrollBy(0, window.innerHeight);")
    time.sleep(pause)
    after = len(driver.find_elements(By.CSS_SELECTOR, CARD_IMG_SELECTOR))
    return after > before


def download_one(session, url, save_dir=SAVE_DIR):
    """URL에서 이미지 1장 다운로드 (MD5 해시 파일명)."""
    r = session.get(url, timeout=15)
    r.raise_for_status()

    ext = Path(urlparse(url).path).suffix or ".jpg"
    name = hashlib.md5(url.encode()).hexdigest()[:12] + ext
    path = save_dir / name

    if path.exists():
        return name, False

    path.write_bytes(r.content)
    return name, True


def crawl_row_by_row(driver, save_dir=SAVE_DIR, max_count=MAX_IMAGES, row_size=ROW_SIZE):
    """위에서부터 한 줄(4개)씩 다운로드, 없을 때만 스크롤."""
    if "Access Denied" in driver.title:
        raise RuntimeError("오늘의집 접근 차단됨. headless 대신 최소화 창 모드로 실행 중인지 확인하세요.")

    session = requests.Session()  # 연결 재사용으로 다운로드 속도 향상
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": "https://ohou.se/",
    })

    downloaded = set()  # 이미 처리한 URL
    saved = 0
    skipped = 0
    row_num = 0
    no_new_scrolls = 0  # 연속 스크롤 실패 횟수

    while saved < max_count:
        pending = get_pending_urls(driver, downloaded)

        if not pending:
            print("다운로드할 이미지 없음 → 스크롤")
            if not scroll_once(driver):
                no_new_scrolls += 1
                if no_new_scrolls >= 3:  # 3회 연속 새 콘텐츠 없으면 종료
                    print("더 이상 새 콘텐츠가 없습니다.")
                    break
            else:
                no_new_scrolls = 0
            continue

        no_new_scrolls = 0
        row_num += 1
        batch = pending[:row_size]  # 한 줄 = ROW_SIZE장
        print(f"\n[{row_num}줄] {len(batch)}장 처리")

        for url in batch:
            if saved >= max_count:
                break
            try:
                name, is_new = download_one(session, url, save_dir)
                downloaded.add(url)
                if is_new:
                    saved += 1
                    print(f"  저장: {name} ({saved}/{max_count})")
                else:
                    skipped += 1
                    print(f"  건너뜀(이미 있음): {name}")
            except Exception as e:
                downloaded.add(url)
                print(f"  실패: {url[:60]}... ({e})")

    print(f"\n신규 {saved}장 저장, {skipped}장 건너뜀 → {save_dir.resolve()}")

In [21]:
# === 실행 ===
driver = create_driver()  # Chrome 최소화 모드 (headless는 Access Denied)

try:
    driver.get(SEARCH_URL)  # 오늘의집 피드 열기
    wait_for_cards(driver)  # 카드 로딩 대기
    print(f"페이지 로드 완료: {driver.title}")
    crawl_row_by_row(driver)  # 위→아래 한 줄씩 다운로드
finally:
    driver.quit()  # 브라우저 종료

페이지 로딩 대기 중... (최대 30초)
카드 16개 확인
페이지 로드 완료: 오늘의집 유저들의 일상을 담은 콘텐츠 | 라이프스타일 슈퍼앱, 오늘의집

[1줄] 4장 처리
  저장: 35d3d028eb6e.jpeg (1/100)
  저장: c2cd7da2cf3d.jpeg (2/100)
  저장: e5db091dfd7b.jpeg (3/100)
  저장: 21c17bce67ca.jpg (4/100)

[2줄] 4장 처리
  저장: e9b0bdee38f2.jpg (5/100)
  저장: a1dfe3ea50fb.jpeg (6/100)
  저장: 62d80d1ff7a8.JPG (7/100)
  저장: 9b1325fdb211.png (8/100)

[3줄] 4장 처리
  저장: 921c4715fe3f.jpeg (9/100)
  저장: 3a5ad9150b02.jpg (10/100)
  저장: 71c6eaf3a079.jpg (11/100)
  저장: 757741c5dcd7.jpeg (12/100)

[4줄] 4장 처리
  저장: 430ccec1c47b.jpg (13/100)
  저장: 91d065e69b71.jpg (14/100)
  저장: 1bc9d32396d7.JPG (15/100)
  저장: 25f37462f358.JPG (16/100)
다운로드할 이미지 없음 → 스크롤

[5줄] 4장 처리
  저장: 7968f70bdb8f.jpg (17/100)
  저장: cd8ff027c02f.JPG (18/100)
  저장: 2509c56db69d.JPG (19/100)
  저장: 01c706b7aedb.jpg (20/100)

[6줄] 4장 처리
  저장: c3e55ac06b89.jpeg (21/100)
  저장: f9a18eef40dc.jpg (22/100)
  저장: 5de8e39ee10e.jpg (23/100)
  저장: 03a6878d5ac7.jpeg (24/100)
다운로드할 이미지 없음 → 스크롤

[7줄] 4장 처리
  저장: 445f6b5902d1.jpg (25